In [2]:
from datasets import load_dataset

level_1_problems = load_dataset("gaia-benchmark/GAIA", "2023_level1", split="validation")
print(f"Number of level 1 problems: {len(level_1_problems)}")

Number of level 1 problems: 53


## Define the response format

In [3]:
from pydantic import BaseModel
class GaiaOutput(BaseModel):
    is_solvable: bool
    unsolvable_reason: str = ""
    final_answer: str = ""

## Building the evaluation pipeline

### Setup the system prompt

In [4]:
SYS_PROMPT = """You are a general AI assistant. 
I will ask you a question. Report your thoughts in "thought_process" and finish your answer in "final_answer".
YOUR FINAL ANSWER should be a number OR as few words as possible OR a comma separated list of numbers and/or strings. 
If you are asked for a number, don't use comma to write your number neither use units such as $ or percent sign unless specified otherwise. 
If you are asked for a string, don't use articles, neither abbreviations (e.g. for cities), and write the digits in plain text unless specified otherwise. 
If you are asked for a comma separated list, apply the above rules depending of whether the element to be put in the list is a number or a string.
"""

### Definiere die Modelle und ihre API-Limits


In [5]:
import asyncio

PROVIDER_SEMAPHORES = {
    "ollama/qwen3": asyncio.Semaphore(10),
    "ollama/gemma3": asyncio.Semaphore(10),
}

In [6]:
from litellm import acompletion

async def solve_problem(model: str, question: str) -> GaiaOutput:
    """ Solve a single problem and return structured output"""
    async with PROVIDER_SEMAPHORES[model]:
        response = await acompletion(
            model=model,
            messages=[
                {"role": "system", "content": SYS_PROMPT},
                {"role": "user","content": question},
            ], 
            response_format=GaiaOutput,
            num_retries=2,           
        )
        finish_reason = response.choices[0].finish_reason
        content = response.choices[0].message.content

        if finish_reason == "refusal" or content is None:
            return GaiaOutput(
                is_solvable=False,
                unsolvable_reason=f"Model refused to answer: {finish_reason}",
                final_answer="",
            )
        return GaiaOutput.model_validate_json(content)

In [7]:
def is_correct(prediction: str | None, answer: str) -> bool:
    if prediction is None:
        return False
    return prediction.strip().lower() == answer.strip().lower()
   

In [8]:
async def evaluate_gaia_single(problem: dict, model: str) -> dict:
    """Evaluate a single problem-model pair and return the result"""
    try:
        output = await solve_problem(model, problem["Question"])
        return {
            "task_id": problem["task_id"],
            "model": model,
            "correct": is_correct(output.final_answer, problem["Final answer"]),
            "is_solvable": output.is_solvable,
            "prediction": output.final_answer,
            "answer": problem["Final answer"],
            "unsolvable_reason": output.unsolvable_reason,
        }
    except Exception as e:
        return {
            "task_id": problem["task_id"],
            "model": model,
            "correct": False,
            "is_solvable": None,
            "prediction": None,
            "answer": problem["Final answer"],
            "error": str(e)
        }

In [9]:
from tqdm.asyncio import tqdm_asyncio

async def run_experiment(
    problems: list[dict],
    models: list[str],    
) -> dict[str, list]: 
    """Evalueate all models on all problems."""
    tasks = [
        evaluate_gaia_single(problem, model)
        for problem in problems
        for model in models
    ]

    print(tasks)

    all_results = await tqdm_asyncio.gather(*tasks)

    results = {model: [] for model in models}
    for result in all_results:
        results[result["model"]].append(result)
    
    return results;

In [ ]:
MODELS = [
 "ollama/gemma3",
 # "ollama/:20b"
 # "ollama/qwen3"
]

subset = level_1_problems.select(range(20))
results = await run_experiment(subset, MODELS)

[<coroutine object evaluate_gaia_single at 0x7fa31077e130>, <coroutine object evaluate_gaia_single at 0x7fa31077f890>, <coroutine object evaluate_gaia_single at 0x7fa31077f670>, <coroutine object evaluate_gaia_single at 0x7fa31077fcd0>, <coroutine object evaluate_gaia_single at 0x7fa31077fef0>, <coroutine object evaluate_gaia_single at 0x7fa31077fde0>, <coroutine object evaluate_gaia_single at 0x7fa31077f9a0>, <coroutine object evaluate_gaia_single at 0x7fa310704040>, <coroutine object evaluate_gaia_single at 0x7fa310704150>, <coroutine object evaluate_gaia_single at 0x7fa310704260>, <coroutine object evaluate_gaia_single at 0x7fa310704370>, <coroutine object evaluate_gaia_single at 0x7fa310704480>, <coroutine object evaluate_gaia_single at 0x7fa310704590>, <coroutine object evaluate_gaia_single at 0x7fa3107046a0>, <coroutine object evaluate_gaia_single at 0x7fa3107047b0>, <coroutine object evaluate_gaia_single at 0x7fa310704ae0>, <coroutine object evaluate_gaia_single at 0x7fa310704bf

100%|██████████| 20/20 [00:00<00:00, 108801.66it/s]


In [19]:
results

{'ollama/gpt-oss:20b': [{'task_id': 'e1fc63a2-da7a-432f-be78-7c4a95598703',
   'model': 'ollama/gpt-oss:20b',
   'correct': False,
   'is_solvable': None,
   'prediction': None,
   'answer': '17',
   'error': "'ollama/gpt-oss:20b'"},
  {'task_id': '8e867cd7-cff9-4e6c-867a-ff5ddc2550be',
   'model': 'ollama/gpt-oss:20b',
   'correct': False,
   'is_solvable': None,
   'prediction': None,
   'answer': '3',
   'error': "'ollama/gpt-oss:20b'"},
  {'task_id': 'ec09fa32-d03f-4bf8-84b0-1f16922c3ae4',
   'model': 'ollama/gpt-oss:20b',
   'correct': False,
   'is_solvable': None,
   'prediction': None,
   'answer': '3',
   'error': "'ollama/gpt-oss:20b'"},
  {'task_id': '5d0080cb-90d7-4712-bc33-848150e917d3',
   'model': 'ollama/gpt-oss:20b',
   'correct': False,
   'is_solvable': None,
   'prediction': None,
   'answer': '0.1777',
   'error': "'ollama/gpt-oss:20b'"},
  {'task_id': 'a1e91b78-d3d8-4675-bb8d-62741b4b68a6',
   'model': 'ollama/gpt-oss:20b',
   'correct': False,
   'is_solvable': N

In [13]:

[r for r in results['ollama/gpt-oss:120b-cloud'] if r['correct']]



KeyError: 'ollama/gpt-oss:120b-cloud'